# Initial modeling

This notebook orchestrates the modeling pipeline for the **VentureSurvive** project using the functions defined in `src/`.

In [25]:
# Main imports for the modeling pipeline

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import sys
from pathlib import Path

# Add the project root to the Python path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root added:", project_root)


from src import data as data_mod
from src import features as features_mod
from src import split as split_mod
from src import model as model_mod
from src import evaluate as eval_mod

import importlib
from src import data as data_mod

importlib.reload(data_mod)  # recharge le fichier src/data.py
print("DATA_PATH_DEFAULT reloaded =", data_mod.DATA_PATH_DEFAULT)


Project root added: /Users/yusefbag/CascadeProjects/venturesurvive
DATA_PATH_DEFAULT reloaded = /Users/yusefbag/CascadeProjects/venturesurvive/data/startups_raw.csv


## Data loading and preparation

In this section, we load the raw data and apply the first cleaning steps: status filtering, label creation, date conversion, and funding amount cleaning.

In [26]:
# Loading raw data
# Load the raw data via the utility function from the data module

df_raw = data_mod.load_raw()
print("df_raw.shape:", df_raw.shape)

# Filtering statuses
df_filtered = data_mod.filter_status(df_raw)
print("df_filtered.shape (after status filtering):", df_filtered.shape)

# Creating the binary success label
df_labeled = data_mod.create_label(df_filtered)

# Converting dates
df_dates = data_mod.convert_dates(df_labeled)

# Cleaning the funding_total_usd column
df_clean = data_mod.clean_funding(df_dates)

print("df_clean.shape (after cleaning):", df_clean.shape)

df_clean.head()

df_raw.shape: (66368, 14)
df_filtered.shape (after status filtering): (13334, 14)
df_clean.shape (after cleaning): (13334, 15)


,permalink,name,homepage_url,category_list,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,first_funding_at,last_funding_at,success
15,/organization/1-mainstream,1 Mainstream,http://www.1mainstream.com,Apps|Cable|Distribution|Software,5000000.0,acquired,USA,CA,SF Bay Area,Cupertino,1,2012-03-01,2015-03-17,2015-03-17,1
20,/organization/1000-markets,1000 Markets,http://www.1000markets.com,Art|E-Commerce|Marketplaces,500000.0,acquired,USA,WA,Seattle,Seattle,1,2009-01-01,2009-05-15,2009-05-15,1
23,/organization/1000memories,1000memories,http://1000memories.com,Curated Web,2535000.0,acquired,USA,CA,SF Bay Area,San Francisco,2,2010-07-01,2010-01-01,2011-02-16,1
31,/organization/100plus,100Plus,http://www.100plus.com,Analytics,1250000.0,acquired,USA,CA,SF Bay Area,San Francisco,2,2011-09-16,2011-11-02,2011-11-30,1
32,/organization/1010data,1010data,http://www.1010data.com,Software,35000000.0,acquired,USA,NY,New York City,New York,1,2000-01-01,2010-03-08,2010-03-08,1


In [27]:
# Saving cleaned dataset for reuse

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
clean_path = processed_dir / "startups_clean.csv"

df_clean.to_csv(clean_path, index=False)
print("Cleaned dataset saved to:", clean_path)

Cleaned dataset saved to: /Users/yusefbag/CascadeProjects/venturesurvive/data/processed/startups_clean.csv


## Feature construction

In this section, we build time-based, geographic, and category features, then assemble a feature DataFrame ready for modeling.

In [28]:
# Assembling features from the cleaned DataFrame

# Build the full feature set
df_features = features_mod.assemble_features(df_clean)

# Clean separation of target and features
y = df_features["success"].astype(int)
X_full = df_features.drop(columns=["success"])

# Keep only numeric and boolean columns for the model
X_numeric = X_full.select_dtypes(include=["number", "bool"]).copy()

# Convert boolean columns to float for scikit-learn compatibility
bool_cols = X_numeric.select_dtypes(include=["bool"]).columns
if len(bool_cols) > 0:
    X_numeric[bool_cols] = X_numeric[bool_cols].astype(float)

numeric_feature_names = list(X_numeric.columns)

print("df_features.shape:", df_features.shape)
print("Total number of features (excluding target):", X_full.shape[1])
print("Number of numeric/boolean features used:", len(numeric_feature_names))
print("Target distribution (success):\n", y.value_counts())

X_numeric.head()

df_features.shape: (13334, 22)
Total number of features (excluding target): 21
Number of numeric/boolean features used: 12
Target distribution (success):
 success
1    7096
0    6238
Name: count, dtype: int64


,funding_total_usd,funding_rounds,age_at_first_funding,time_between_first_last,is_us,is_uk,is_eu,country_code_missing,state_code_missing,region_missing,city_missing,log_funding_total
15,5000000.0,1,1111.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,15.424949
20,500000.0,1,134.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,13.122365
23,2535000.0,2,-181.0,411.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,14.745705
31,1250000.0,2,47.0,28.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,14.038655
32,35000000.0,1,3719.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,17.370859


## Train / test split

In this section, we perform a temporal train/test split based on `first_funding_at`, then optionally scale numeric features.

In [29]:
# Temporal train/test split using numeric features only

# Keep the full df_features to have date columns available for the temporal split
df_for_split = df_features.copy()

# Temporal split using the utility function
train_df, test_df = split_mod.temporal_split(
    df_for_split, cutoff_date="2013-01-01", date_col="first_funding_at"
)

print("train_df.shape:", train_df.shape)
print("test_df.shape:", test_df.shape)

# Split X / y using only numeric/boolean columns
feature_cols = numeric_feature_names

y_train = train_df["success"].astype(int)
y_test = test_df["success"].astype(int)

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

# Explicitly convert any remaining boolean columns to float (safety)
bool_cols_train = X_train.select_dtypes(include=["bool"]).columns
if len(bool_cols_train) > 0:
    X_train[bool_cols_train] = X_train[bool_cols_train].astype(float)
    X_test[bool_cols_train] = X_test[bool_cols_train].astype(float)

# Scale numeric features only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)
print("Number of features used:", len(feature_cols))

train_df.shape: (10688, 22)
test_df.shape: (2646, 22)
X_train_scaled shape: (10688, 12)
X_test_scaled shape: (2646, 12)
Number of features used: 12


## Model training

In this section, we prepare the training of baseline models (Logistic Regression and Random Forest), but **they are not run until you manually execute the cells**.

In [30]:
# Logistic Regression training and evaluation with numeric features only

from sklearn.metrics import brier_score_loss

# Train baseline logistic regression model on scaled numeric features
logistic_model = model_mod.train_logistic(X_train_scaled, y_train.values)

# Standard classification metrics (including ROC AUC when available)
logistic_results = eval_mod.evaluate_classification(
    logistic_model, X_test_scaled, y_test.values
)

# Compute Brier score using predicted probabilities when available
if hasattr(logistic_model, "predict_proba"):
    y_prob = logistic_model.predict_proba(X_test_scaled)[:, 1]
    logistic_brier = brier_score_loss(y_test.values, y_prob)
else:
    logistic_brier = float("nan")

logistic_metrics = {
    "accuracy": float(logistic_results.get("accuracy", float("nan"))),
    "roc_auc": float(logistic_results.get("roc_auc", float("nan"))),
    "brier": float(logistic_brier),
}

print("Logistic Regression results:", logistic_results)
print("Brier score:", logistic_brier)

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## Évaluation et visualisations

Dans cette section, nous proposons des cellules pour tracer les courbes ROC / PR et la matrice de confusion, une fois les modèles entraînés.

In [ ]:
# Final summary of the modeling pipeline run (numeric features only)

summary = {
    "numeric_features": numeric_feature_names,
    "n_numeric_features": len(numeric_feature_names),
    "train_shape": train_df.shape,
    "test_shape": test_df.shape,
    "logistic_accuracy": logistic_metrics.get("accuracy"),
    "logistic_roc_auc": logistic_metrics.get("roc_auc"),
    "logistic_brier": logistic_metrics.get("brier"),
    "clean_data_path": str(clean_path),
}

print("=== Modeling pipeline summary ===")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\nComment:")
print("This baseline uses only numeric features (and boolean features converted to float).")
print("Categorical columns were intentionally excluded to avoid type errors ")
print("(for example 'could not convert string to float').")

print("\nNext steps suggestions:")
print("- Consider encoding categorical variables (one-hot, target encoding, etc.).")
print("- Try other models (Random Forest, Gradient Boosting, etc.).")
print("- Refine feature selection and hyperparameters based on these baseline results.")

NameError: name 'logistic_metrics' is not defined

In [ ]:
# Exemples d'utilisation des fonctions d'évaluation (à exécuter après entraînement des modèles)

# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# eval_mod.plot_roc(logistic_model, X_test_scaled.values, y_test.values, ax=axes[0])
# eval_mod.plot_pr(logistic_model, X_test_scaled.values, y_test.values, ax=axes[1])
# eval_mod.plot_confusion(logistic_model, X_test_scaled.values, y_test.values, normalize=True, ax=axes[2])
# plt.tight_layout()

# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# eval_mod.plot_roc(rf_model, X_test_scaled.values, y_test.values, ax=axes[0])
# eval_mod.plot_pr(rf_model, X_test_scaled.values, y_test.values, ax=axes[1])
# eval_mod.plot_confusion(rf_model, X_test_scaled.values, y_test.values, normalize=True, ax=axes[2])
# plt.tight_layout()